# Tahap 5 + 6 CBR: Revise/Retain + Evaluation
### CBR Putusan Wanprestasi

**Tim:**
- Ahmad Qayyim — 202310370311286
- Bintang Mars Satria Tuhu — 202310370311410

---

## Tujuan

**Tahap 5 (Revise & Retain):**
- Fungsi `retain_case(new_case, verified)` untuk menambah kasus baru ke case base
- Demo 3 skenario: valid+verified / tidak verified / duplicate

**Tahap 6 (Evaluation):**
- Retrieval metrics: Precision@k, Recall@k, F1@k, Hit Rate@k, MRR, Top-1 Accuracy
- Prediction metrics: Accuracy, Precision, Recall, F1 (weighted + macro)
- Confusion matrix
- Error analysis kritis

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/kuliah/smt 6/Penalaran Komputer/Tugas SUB-CPMK 3/cbr-putusan-wanprestasi')
os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
import sys, importlib.util
sys.path.insert(0, str(PROJECT_DIR / 'src'))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

retain_mod = load_module('retain_mod', PROJECT_DIR / 'src' / '06_retain.py')
eval_mod   = load_module('eval_mod', PROJECT_DIR / 'src' / '07_evaluation.py')

# Tahap 5: Revise & Retain

## 2. Demo retain_case() Workflow

In [ ]:
demo_results = retain_mod.demo_retain_workflow()
print()
for r in demo_results:
    print(f"Skenario [{r['skenario']:20s}]  →  status: {r['status']}")
    print(f"   message: {r['message']}")
    print()

**Penjelasan workflow:**

1. **Valid + Verified → RETAINED**: Kasus baru lolos validasi dan ditambah ke case base.
2. **Tidak Verified → REJECTED**: Sistem menolak menambah kasus yang belum diverifikasi manual.
3. **Duplicate case_id → DUPLICATE**: Sistem mencegah duplikasi.

Demo cases otomatis di-cleanup setelah demo agar tidak mencemari case base untuk evaluasi.

# Tahap 6: Model Evaluation

## 3. Run Evaluation Pipeline

In [ ]:
eval_result = eval_mod.run()

## 4. Retrieval Metrics

In [ ]:
import pandas as pd

retrieval_df = pd.read_csv('data/eval/retrieval_metrics.csv')
print('Per-query retrieval metrics:')
print(retrieval_df.to_string(index=False))
print()

avg = retrieval_df[retrieval_df['query_id'] == 'AVERAGE'].iloc[0]
print('=== RATA-RATA ===')
print(f"  Precision@5    : {avg['precision_at_k']:.4f}")
print(f"  Recall@5       : {avg['recall_at_k']:.4f}")
print(f"  F1@5           : {avg['f1_at_k']:.4f}")
print(f"  Hit Rate@5     : {avg['hit_rate_at_k']:.4f}")
print(f"  MRR            : {avg['reciprocal_rank']:.4f}")
print(f"  Top-1 Accuracy : {avg['top1_accuracy']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

avg = retrieval_df[retrieval_df['query_id'] == 'AVERAGE'].iloc[0]
metrics_names = ['precision_at_k', 'recall_at_k', 'f1_at_k', 'hit_rate_at_k', 'reciprocal_rank', 'top1_accuracy']
metrics_label = ['Precision@5', 'Recall@5', 'F1@5', 'Hit Rate@5', 'MRR', 'Top-1 Acc']
values = [avg[m] for m in metrics_names]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(metrics_label, values, color='steelblue', edgecolor='black')
ax.set_ylabel('Score')
ax.set_title('Retrieval Performance Metrics (averaged over synthetic queries)')
ax.set_ylim(0, 1.15)
for b, v in zip(bars, values):
    ax.text(b.get_x()+b.get_width()/2, v+0.03, f'{v:.3f}', ha='center', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('reports/figures/retrieval_metrics.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Prediction Metrics — Side-by-Side Comparison

In [ ]:
pred_metrics_df = pd.read_csv('data/eval/prediction_metrics.csv')
print('Prediction metrics comparison:')
print(pred_metrics_df.to_string(index=False))

In [ ]:
import numpy as np

metrics_cols = ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']

fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(metrics_cols))
w = 0.35

for i, (_, row) in enumerate(pred_metrics_df.iterrows()):
    offset = (i - 0.5) * w
    color = ['steelblue', 'coral'][i]
    bars = ax.bar(x + offset, [row[m] for m in metrics_cols], w,
                  label=row['name'], color=color, edgecolor='black')
    for b, m in zip(bars, metrics_cols):
        ax.text(b.get_x()+b.get_width()/2, row[m]+0.02,
                f"{row[m]:.3f}", ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', '\n') for m in metrics_cols])
ax.set_ylabel('Score')
ax.set_title('Prediction Metrics: Majority Vote vs Weighted Similarity')
ax.set_ylim(0, 1.15)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('reports/figures/prediction_metrics.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

pred_df = pd.read_csv('data/results/predictions.csv')
syn = pred_df[pred_df['source'] == 'synthetic']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for i, method in enumerate(syn['prediction_method'].unique()):
    sub = syn[syn['prediction_method'] == method]
    y_true = sub['actual_solution'].values
    y_pred = sub['predicted_solution'].values
    labels = sorted(set(list(y_true) + list(y_pred)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=axes[i])
    axes[i].set_title(f'{method}')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')
    axes[i].tick_params(axis='x', rotation=20)
    axes[i].tick_params(axis='y', rotation=0)

plt.suptitle('Confusion Matrix per Method (synthetic queries)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('reports/figures/confusion_matrix_final.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Error Analysis

In [ ]:
error_df = pd.read_csv('data/eval/error_analysis.csv')
print(f'Total error cases: {len(error_df)}\n')
if len(error_df) > 0:
    print(error_df[['query_id', 'source', 'method', 'actual', 'predicted', 'confidence', 'error_types']].to_string(index=False))

In [ ]:
# Breakdown jenis error
from collections import Counter

if len(error_df) > 0:
    all_types = []
    for et_str in error_df['error_types']:
        all_types.extend(et_str.split(', '))
    type_counter = Counter(all_types)
    
    fig, ax = plt.subplots(figsize=(9, 4))
    types = [t for t, _ in type_counter.most_common()]
    counts = [c for _, c in type_counter.most_common()]
    
    bars = ax.barh(types, counts, color='#e74c3c', edgecolor='black')
    ax.set_xlabel('Frekuensi')
    ax.set_title('Breakdown Jenis Error pada Prediksi')
    for b, v in zip(bars, counts):
        ax.text(v + 0.1, b.get_y()+b.get_height()/2, str(v), va='center', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig('reports/figures/error_breakdown.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('Tidak ada error untuk dianalisis.')

## 8. Diskusi Kritis & Rekomendasi Perbaikan

### Temuan Utama

**Retrieval (TF-IDF + Cosine):**
- Performa retrieval **sangat baik** pada synthetic queries (semua case sumber muncul di top-5).
- TF-IDF dengan ngram (1,2) + Indonesian stopwords menangkap pola legal dengan baik.

**Solution Reuse:**
- **Weighted Similarity 100% akurat** vs **Majority Vote 80%** pada synthetic queries.
- Weighted method lebih sensitif terhadap ranking similarity, sehingga unggul.

### Keterbatasan Sistem

1. **Class Imbalance Ekstrem**
   - Distribusi case base sangat tidak seimbang: 53% kategori 'Tidak Dapat Diterima'
   - Menyebabkan bias prediksi ke majority class pada query generik (manual queries)

2. **Dataset Size Kecil (32 cases)**
   - Test set hanya ~7 case → metric classifier kurang stabil
   - Beberapa kategori (Dikabulkan Sebagian, Ditolak) under-represented

3. **Kemiripan Vocabulary ≠ Kemiripan Substansi Hukum**
   - TF-IDF menangkap kemiripan kata, bukan kemiripan struktur hukum
   - Dua kasus dengan pasal sama bisa berbeda outcome

4. **Manual Queries Sulit Diprediksi**
   - Skenario generik (kredit, jual beli) muncul di banyak kategori berbeda
   - Tanpa fitur diskriminatif spesifik, sistem cenderung memilih majority class

### Rekomendasi Perbaikan

1. **Tambah ukuran dataset** (target: 100+ cases) untuk distribusi lebih representatif
2. **Class balancing** — pakai oversampling (SMOTE) atau class weighting di classifier
3. **Embedding semantik** (IndoBERT/Sentence-BERT) sebagai alternatif TF-IDF
4. **Hybrid retrieval** — gabung TF-IDF dengan metadata filter (pasal, pengadilan)
5. **Synonym mapping istilah hukum** untuk handle variasi terminologi
6. **Cross-validation** k-fold untuk metric classifier yang lebih reliable
7. **Manual validation** untuk query manual sebagai ground truth eksternal

## 9. Final Checklist Tahap 5+6

In [ ]:
from pathlib import Path

checks = {
    'retain_case() function works'      : callable(retain_mod.retain_case),
    'logs/retain.log created'           : Path('logs/retain.log').exists(),
    'retrieval_metrics.csv generated'    : Path('data/eval/retrieval_metrics.csv').exists(),
    'prediction_metrics.csv generated'   : Path('data/eval/prediction_metrics.csv').exists(),
    'error_analysis.csv generated'       : Path('data/eval/error_analysis.csv').exists(),
    'logs/evaluation.log created'        : Path('logs/evaluation.log').exists(),
    'Visualisasi tersimpan (4 figures)'  : all(Path(f'reports/figures/{f}').exists() for f in
                                                ['retrieval_metrics.png', 'prediction_metrics.png',
                                                 'confusion_matrix_final.png', 'error_breakdown.png']),
}

for k, v in checks.items():
    print(f'  [{"OK" if v else "FAIL"}]  {k}')

if all(checks.values()):
    print('\nSTATUS: SIAP lanjut ke Tahap 7 (README + Final Commit)')
else:
    print('\nSTATUS: Ada checklist yang gagal, cek output di atas.')